# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

A model score is not a product. This notebook turns the validated Week-5 model into something a person can
act on Monday morning: a ranked queue, a reason code per page, an action per reason, an effort estimate, a
review gate, and the limits written down next to the output.

**What this is.** A research artifact — a notebook, two data files and four figures, all regenerated by running
it. It is not a service, it has no scheduler, it changes nothing on its own, and every row it exports carries
`auto_apply_allowed = False`.

**The division of labour, stated up front.** The **model ranks** (it is the only thing that sees all eight
features at once). The **rules explain and prescribe** (reason codes and actions are deterministic,
inspectable, and can be argued with). A 0.64-AUC ranker earns the right to order a queue; it has not earned
the right to decide what to do with a page.

## 1. Ranked actions + reason codes

The queue has to answer three questions for every row: **why is this page here**, **what should someone do
about it**, and **how much work is that**. The model answers only the first, and only indirectly — it gives a
risk score. The reason code is a rule over observable features, so a reviewer can check it without trusting
the model.

**Four primary reason codes** — mutually exclusive, checked in this order, and the only thing that decides the
action. All measured on 2026-03-31, all visible in the warehouse:

| Primary code | Rule | Action it prescribes | Why this rule |
|---|---|---|---|
| `ZERO_CLICKS` | zero March clicks where the peer median predicts **≥ 10** | check tracking and query intent before editing | at 2,000 impressions and a 0.2% peer median, zero clicks is six missing clicks — noise; at 30,000 it is not |
| `CTR_FAR_BELOW_PEERS` | visible (pos 1–20), CTR < **50%** of the tier peer median | snippet rewrite; body refresh too if it is also fading | the frozen ML-07 rule's strongest signal, kept continuous |
| `CTR_BELOW_PEERS` | visible, CTR < **70%** of the peer median | snippet test | the softer half of the same gap |
| `MOMENTUM_FADE` | `log_momentum ≤ −0.35`, i.e. the last ten days of March ran ~30% below the first ten | body refresh | the Week-5 feature, and the one that carried the model |

**Four secondary markers** — they never decide the action; they refine the description and the review gate:

| Marker | Rule | What it changes |
|---|---|---|
| `LOW_EXPECTED_CLICKS` | the peer median predicts fewer than **5** clicks for this page | blocks any CTR verdict: 500 impressions at a 0.2% peer median is one expected click, and 0-vs-2 is not evidence |
| `DEEP_POSITION` | average position > 20 | sends the row to the watchlist — a copy rewrite at position 40 is the wrong tool |
| `THIN_VOLUME` | < 500 March impressions | sends the row to the watchlist — below the floor the label is noise |
| `LOW_PERSISTENCE` | impressions on fewer than 20 days | marks a page whose traffic is too intermittent to read as a trend |

**The expected-clicks gate is the one rule I added on top of the frozen baseline**, and it is measured in the
cell below: most of the raw CTR flags sit on pages where the peer median predicts a handful of clicks, so a
CTR ratio there is a coin flip dressed as a percentage. Momentum is an impression ratio, so a fading page
stays actionable at any click count — only the CTR codes need this gate.

**The effort gate is the second.** A two-hour refresh on a page with 700 impressions a month cannot pay for
itself, however clearly the page is fading. So an action that costs two hours or more needs **2,000**
impressions; below that the row becomes `WATCHLIST_EFFORT`. The 500-impression floor stays where it belongs —
on *detection* (the label is noise below it), not on *effort allocation*. The cell below prices both gates:
how much of the queue they remove, and how little of the measured gap those rows actually held.

**Disposition.** Not every scored page deserves an action. `actionable` means visible, ≥500 impressions, and a
primary code fired. `watchlist` means a marker fired but the page is thin, deep, or has too few expected
clicks to judge. `no_action` means the page is healthy on every observable signal. The queue is sorted by
model score; the disposition decides whether a row is worked.

**The batch is not the head of the queue.** §1 measures a problem the model cannot see: the global top 100 is
overwhelmingly one client, because one client's pages are systematically zero-click. A portfolio-global
ranking is therefore not client-fair, so the weekly batch is built by **interleaving** — every client's best
page first, then every client's second, capped at 10 per client and 50 per week. `rank` stays the model's
global order (that is the research artifact); `in_weekly_batch` is what a person actually works.

**Value, honestly.** `click_gap` is *peer CTR × impressions − clicks*, floored at zero: the clicks this page
would have earned at its tier median. It is a **sizing heuristic, not a forecast** — it says how big the gap
is, not how much of it a rewrite recovers.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import duckdb
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"])
    import duckdb
    from huggingface_hub import hf_hub_download

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from threadpoolctl import threadpool_limits

threadpool_limits(limits=1)  # one thread keeps the refit reproducible run to run

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "march": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "april": "fact_content_daily_performance/month=2026-04/data_0.parquet",
    "dim_content": "dim_content.parquet",
}
local = {name: hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)
         for name, path in FILES.items()}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
for name, path in local.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{Path(path).as_posix()}')")

KEYS = ["client_hash_id", "content_hash_id"]

pages = con.sql('''
WITH m AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_30d,
           SUM(gsc_clicks)      AS clicks_30d,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d,
           SUM(CASE WHEN report_date <= DATE '2026-03-10' THEN gsc_impressions ELSE 0 END) AS imp_first10,
           SUM(CASE WHEN report_date >= DATE '2026-03-22' THEN gsc_impressions ELSE 0 END) AS imp_last10
    FROM march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT m.*, c.content_type,
       DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days
FROM m JOIN dim_content c USING (client_hash_id, content_hash_id)
ORDER BY client_hash_id, content_hash_id
''').df()

labels = con.sql('''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
FROM april WHERE gsc_data_available IS TRUE GROUP BY 1, 2
ORDER BY 1, 2
''').df()

pages["ctr"] = 100.0 * pages.clicks_30d / pages.impressions_30d
pages["position_tier"] = pd.cut(pages.avg_position_30d, [0, 3, 10, 20, 50, 10000],
                                labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
pages["visible"] = (pages.avg_position_30d > 0) & (pages.avg_position_30d <= 20)
peer = pages[pages.visible].groupby("position_tier", observed=True).ctr.median()
pages["peer_ctr"] = pages.position_tier.map(peer)
pages["ctr_ratio"] = pages.ctr / pages.peer_ctr
pages["zero_clicks"] = (pages.clicks_30d == 0).astype(int)
pages["log_impressions"] = np.log1p(pages.impressions_30d)
pages["log_momentum"] = np.log((pages.imp_last10 + 1) / (pages.imp_first10 + 1))
pages["click_gap"] = (pages.peer_ctr / 100 * pages.impressions_30d - pages.clicks_30d).clip(lower=0)

FEATS_NUM = ["ctr_ratio", "zero_clicks", "log_impressions", "avg_position_30d",
             "days_with_impressions_30d", "content_age_days", "log_momentum"]
FEATS_CAT = ["content_type"]
FEATURES = FEATS_NUM + FEATS_CAT


def make_hgb(num, cat=FEATS_CAT):
    return Pipeline([
        ("pre", ColumnTransformer([
            ("num", SimpleImputer(strategy="median"), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat),
        ])),
        ("clf", HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_depth=4,
                                               min_samples_leaf=40, random_state=42)),
    ])


# ---- labelled frame: used to fit the final ranker and to measure what happened to each pattern
labelled = pages.merge(labels, on=KEYS, how="inner")
labelled["declined"] = (labelled.imp_apr < 0.8 * labelled.impressions_30d).astype(int)
labelled = labelled[labelled.impressions_30d > 0].reset_index(drop=True)

model = make_hgb(FEATS_NUM).fit(labelled[FEATURES], labelled.declined)

# ---- queue: every page the features can describe, including the 548 with no April row
queue = pages.copy()
queue["model_score"] = model.predict_proba(queue[FEATURES])[:, 1]

VOLUME_FLOOR = 500
EFFORT_FLOOR = 2000
MOMENTUM_CUT = -0.35
CTR_GAP_CUT = 0.70
CTR_SEVERE_CUT = 0.50
MIN_EXPECTED_CLICKS = 5

queue["expected_clicks"] = queue.peer_ctr / 100 * queue.impressions_30d
# a CTR ratio is only readable when the peer median predicts enough clicks for the ratio to mean anything
queue["f_ctr_judgeable"] = queue.expected_clicks >= MIN_EXPECTED_CLICKS
queue["f_low_expected"] = queue.visible & ~queue.f_ctr_judgeable
queue["f_ctr_far"] = queue.visible & queue.f_ctr_judgeable & (queue.ctr_ratio < CTR_SEVERE_CUT)
queue["f_ctr_gap"] = queue.visible & queue.f_ctr_judgeable & (queue.ctr_ratio < CTR_GAP_CUT)
queue["f_momentum"] = queue.log_momentum <= MOMENTUM_CUT
# zero clicks only counts as its own problem when the peer median predicts at least ten
queue["f_zero_click"] = (queue.clicks_30d == 0) & (queue.expected_clicks >= 10)
queue["f_deep"] = queue.avg_position_30d > 20
queue["f_thin"] = queue.impressions_30d < VOLUME_FLOOR


def reason_codes(r):
    '''Primary first (decides the action), then the secondary markers that refine it.'''
    out = []
    if r.f_zero_click:
        out.append("ZERO_CLICKS")
    if r.f_ctr_far:
        out.append("CTR_FAR_BELOW_PEERS")
    elif r.f_ctr_gap:
        out.append("CTR_BELOW_PEERS")
    if r.f_momentum:
        out.append("MOMENTUM_FADE")
    if r.f_low_expected:
        out.append("LOW_EXPECTED_CLICKS")
    if r.f_deep:
        out.append("DEEP_POSITION")
    if r.days_with_impressions_30d < 20:
        out.append("LOW_PERSISTENCE")
    if r.f_thin:
        out.append("THIN_VOLUME")
    return out


queue["reasons"] = [reason_codes(r) for r in queue.itertuples()]
queue["reason_primary"] = [r[0] if r else "NO_FLAG" for r in queue.reasons]
queue["reason_secondary"] = [";".join(r[1:]) for r in queue.reasons]


def core_action(r):
    '''(action_code, action, effort_hours) — deterministic, inspectable, and arguable.'''
    if r.f_thin:
        return ("WATCHLIST", "Watchlist: too little volume to judge. Re-check if impressions recover.", 0.0)
    if r.f_deep and not r.f_ctr_gap:
        return ("INTERNAL_LINKS", "Improve relevance and internal links. Do not rewrite the copy yet.", 1.5)
    if r.f_zero_click:
        return ("INTENT_CHECK",
                "Check tracking and query intent first: zero clicks where the peer median predicts "
                f"{r.expected_clicks:.0f} is a measurement or mismatch question before it is a copy question.",
                0.5)
    if r.f_ctr_far and r.f_momentum:
        return ("REFRESH_SNIPPET", "Refresh the body and rewrite the title/meta against the peer CTR gap.", 3.0)
    if r.f_ctr_far:
        return ("SNIPPET_REWRITE", "Rewrite title/meta only. The body is not the measured problem.", 0.75)
    if r.f_ctr_gap and r.f_momentum:
        return ("REFRESH_BODY", "Refresh the body and the snippet; both signals point the same way.", 2.5)
    if r.f_ctr_gap:
        return ("SNIPPET_TEST", "Test a sharper title/description. Keep the body.", 0.75)
    if r.f_momentum:
        return ("REFRESH_BODY", "Refresh facts, examples and recrawl. CTR is not the measured problem.", 2.0)
    if r.f_low_expected:
        return ("WATCHLIST",
                "Watchlist: too few expected clicks for the CTR ratio to mean anything. "
                "Re-check if volume grows.", 0.0)
    return ("NO_ACTION", "No observable problem on these features.", 0.0)


core = [core_action(r) for r in queue.itertuples()]
queue["hours_if_worked"] = [c[2] for c in core]
prescriptions = []
for r, (code, action, hours) in zip(queue.itertuples(), core):
    if hours >= 2.0 and r.impressions_30d < EFFORT_FLOOR:
        prescriptions.append(("WATCHLIST_EFFORT",
                              f"Watchlist: {r.impressions_30d:,.0f} impressions a month cannot pay for a "
                              f"{hours:.0f}-hour edit. Re-check if volume grows.", 0.0))
    else:
        prescriptions.append((code, action, hours))

queue["action_code"] = [p[0] for p in prescriptions]
queue["action"] = [p[1] for p in prescriptions]
queue["effort_hours"] = [p[2] for p in prescriptions]

queue["disposition"] = np.select(
    [queue.action_code == "NO_ACTION",
     queue.action_code.isin(["WATCHLIST", "WATCHLIST_EFFORT", "INTERNAL_LINKS"])],
    ["no_action", "watchlist"], default="actionable")
queue["auto_apply_allowed"] = False
queue["review_required"] = True
queue["value_per_hour"] = np.where(queue.effort_hours > 0,
                                   queue.click_gap / queue.effort_hours, 0.0)

queue = queue.sort_values(["model_score", "click_gap"], ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["rank_value"] = queue.click_gap.div(queue.effort_hours.replace(0, np.nan)) \
    .rank(ascending=False, method="first").fillna(len(queue)).astype(int)

# ---- the weekly batch: interleave clients, at most 10 per client, 50 per week
CAP_PER_CLIENT = 10
BATCH_SIZE = 50
act = queue[queue.disposition == "actionable"].copy()
act["rank_in_client"] = act.groupby("client_hash_id").cumcount() + 1
batch = (act[act.rank_in_client <= CAP_PER_CLIENT]
         .sort_values(["rank_in_client", "model_score"], ascending=[True, False])
         .head(BATCH_SIZE))
batch_keys = pd.MultiIndex.from_frame(batch[KEYS])
queue["in_weekly_batch"] = pd.MultiIndex.from_frame(queue[KEYS]).isin(batch_keys)
queue = queue.merge(act[KEYS + ["rank_in_client"]], on=KEYS, how="left")

print(f"queue: {len(queue):,} pages | {queue.client_hash_id.nunique()} clients | "
      f"model refit on {len(labelled):,} labelled rows")
print(queue.disposition.value_counts().to_string())
print(f"\nactionable: {int((queue.disposition == 'actionable').sum()):,} pages "
      f"({(queue.disposition == 'actionable').mean():.1%} of the queue)")
print("\nreason-code mix among actionable pages:")
mix = act.reason_primary.value_counts()
print(pd.DataFrame({"pages": mix, "share": (mix / mix.sum()).round(3)}).to_string())

ctr_flag = queue.visible & (queue.ctr_ratio < CTR_GAP_CUT)
readable = ctr_flag & queue.f_ctr_judgeable
print(f"\nCTR flags vs readable CTR: {int(ctr_flag.sum()):,} visible pages sit below "
      f"{CTR_GAP_CUT:.0%} of their peer median.")
print(f"  only {int(readable.sum()):,} ({readable.sum() / max(int(ctr_flag.sum()), 1):.1%}) have "
      f">= {MIN_EXPECTED_CLICKS} expected clicks; the rest carry "
      f"{queue.loc[ctr_flag & ~queue.f_ctr_judgeable, 'click_gap'].sum():,.0f} of the "
      f"{queue.loc[ctr_flag, 'click_gap'].sum():,.0f} clicks at stake in that group.")
print("  a 0-vs-2-click difference is not evidence: those rows go to the watchlist, not to an editor.")

gated = queue[queue.action_code == "WATCHLIST_EFFORT"]
print(f"\neffort gate: {len(gated):,} pages would have taken a two-hour-plus edit "
      f"({gated.impressions_30d.median():,.0f} median impressions).")
print(f"  they hold {gated.click_gap.sum():,.0f} of the {queue.click_gap.sum():,.0f} clicks at stake in the "
      f"queue and would have cost {gated.hours_if_worked.sum():,.0f} hours.")
print("  a two-hour refresh on a 700-impression page cannot pay for itself: those rows are watchlist too.")

print("\nwhere the model's own top 100 lands:")
conc = queue.head(100).client_hash_id.value_counts()
print(f"  distinct clients {len(conc)} | largest {conc.iloc[0]}/100 ({conc.iloc[0]}%) | "
      f"top 3 {conc.head(3).sum()}/100")
print("  one client supplies most of the head because it has the most zero-click pages, and")
print("  zero-click is the strongest feature the model has. Hence the interleaved batch below.")

cols = ["rank", "rank_in_client", "client_hash_id", "content_hash_id", "model_score",
        "action_code", "reason_primary", "impressions_30d", "ctr", "peer_ctr", "click_gap"]
print("\nthe weekly batch as a reviewer receives it (first 12 of 50):")
head = batch.head(12)[cols].copy()
head["client_hash_id"] = head.client_hash_id.str[:14] + ".."
head["content_hash_id"] = head.content_hash_id.str[:14] + ".."
print(head.round(2).to_string(index=False))

print(f"\nbatch: {len(batch)} actions | {batch.client_hash_id.nunique()} clients | "
      f"{batch.effort_hours.sum():.1f} hours | {batch.click_gap.sum():,.0f} clicks of measured gap")
print("action mix in the batch:")
print(batch.action_code.value_counts().to_string())

queue: 101,441 pages | 44 clients | model refit on 100,893 labelled rows
disposition
watchlist     81137
no_action     11909
actionable     8395

actionable: 8,395 pages (8.3% of the queue)

reason-code mix among actionable pages:
                     pages  share
reason_primary                   
CTR_FAR_BELOW_PEERS   3493  0.416
MOMENTUM_FADE         3084  0.367
CTR_BELOW_PEERS       1753  0.209
ZERO_CLICKS             65  0.008

CTR flags vs readable CTR: 32,621 visible pages sit below 70% of their peer median.
  only 5,314 (16.3%) have >= 5 expected clicks; the rest carry 24,273 of the 80,307 clicks at stake in that group.
  a 0-vs-2-click difference is not evidence: those rows go to the watchlist, not to an editor.

effort gate: 6,682 pages would have taken a two-hour-plus edit (950 median impressions).
  they hold 4,482 of the 88,656 clicks at stake in the queue and would have cost 13,366 hours.
  a two-hour refresh on a 700-impression page cannot pay for itself: those rows are w

### What the queue looks like

Three things a reviewer can see immediately: the **disposition split** (how much of the queue is actually
work), the **reason mix** (which problem dominates), and the fact that the head is **not** a list of the
biggest pages. That last point is the Week-5 finding carried into the product: the rule's tie-break pulled
large pages to the top, while the model ranks by probability of decline. The playbook keeps both orders —
`rank` for risk and `rank_value` for clicks-per-hour — and §2 measures how much they disagree.

### Archetypes, and what happened to each one last month

A reason code says *which signal fired*. An archetype says *what kind of page this is* — and a playbook needs
both, because "visible page-1 asset" and "deep long-tail page" deserve different treatment even when the same
flag fires.

The archetype is built from three observable bands (position, volume, momentum), which is a deliberate choice:
it can be explained to an editor in one sentence and recomputed from the same warehouse columns every month.
The honest columns are the last three — the **observed** April decline rate of that archetype, the mean
measured click gap across the pages that could be sized at all, and the share that could be sized. The decline
rate is a base rate for calibration, not a promise; the gap is a sizing heuristic, not a forecast.

**The archetype does not rank, and the table shows why.** Every band lands within a few points of the 51.5%
base rate, so "page 1 / high volume" is not inherently safer than "striking / mid volume" — the spread is
composition, not destiny. Archetype answers *what to do with a page*; the model and the reason codes answer
*which page first*.

In [2]:
def archetype(r):
    if r.impressions_30d < VOLUME_FLOOR:
        return "thin volume"
    if r.avg_position_30d > 20:
        return "deep position"
    if r.avg_position_30d <= 10:
        return "page 1 / high volume" if r.impressions_30d >= 5000 else "page 1 / mid volume"
    return "striking / high volume" if r.impressions_30d >= 5000 else "striking / mid volume"


queue["archetype"] = [archetype(r) for r in queue.itertuples()]

# join the observed outcome where a label exists, so each archetype carries its own base rate
outcome = labelled.set_index(KEYS).declined
queue = queue.set_index(KEYS)
queue["observed_declined"] = outcome.reindex(queue.index)
queue = queue.reset_index()

ARCH_ORDER = ["page 1 / high volume", "page 1 / mid volume", "striking / high volume",
              "striking / mid volume", "deep position", "thin volume"]
ARCH_ACTION = {
    "page 1 / high volume": "Protect first: snippet test, then body refresh. Highest click gap per edit.",
    "page 1 / mid volume": "Snippet test or body refresh; cheap to try, easy to measure.",
    "striking / high volume": "Refresh body + snippet; the biggest single-step upside in the portfolio.",
    "striking / mid volume": "Refresh body, or internal links if the gap is not CTR-shaped.",
    "deep position": "Relevance and internal links first. A copy rewrite at position 40 is wasted effort.",
    "thin volume": "Watchlist only. Do not spend an hour on a page with 200 impressions.",
}

arch = queue.groupby("archetype").agg(
    pages=("content_hash_id", "size"),
    actionable=("disposition", lambda s: int((s == "actionable").sum())),
    fading_share=("f_momentum", "mean"),
    observed_decline_rate=("observed_declined", "mean"),
    mean_click_gap=("click_gap", "mean"),
    pct_sized=("click_gap", lambda s: s.notna().mean()),
    median_impressions=("impressions_30d", "median"),
).reindex(ARCH_ORDER)
arch["share_of_queue"] = (arch.pages / arch.pages.sum()).round(3)
arch["mapped_action"] = [ARCH_ACTION[a] for a in arch.index]

print("archetype -> action, with the observed April outcome as a base rate:")
print(arch[["pages", "share_of_queue", "actionable", "fading_share", "observed_decline_rate",
            "mean_click_gap", "pct_sized"]].round(3).to_string())
print("\nmean_click_gap is over the pages whose tier has a peer median (pct_sized); deep tiers have none.")
spread = arch.observed_decline_rate
print(f"\nspread across archetypes: {spread.min():.1%} to {spread.max():.1%} around a "
      f"{float(labelled.declined.mean()):.1%} base rate.")
print("  the archetype says what to do with a page, not how urgent it is — the ranking has to come")
print("  from the model and the reason codes, because the bands barely separate on their own.")
print("\nmapped action per archetype:")
for name, row in arch.iterrows():
    print(f"  {name:<24} n={int(row.pages):>6,}  ->  {row.mapped_action}")

archetype -> action, with the observed April outcome as a base rate:
                        pages  share_of_queue  actionable  fading_share  observed_decline_rate  mean_click_gap  pct_sized
archetype                                                                                                                
page 1 / high volume     8898           0.088        3315         0.152                  0.445           5.492      1.000
page 1 / mid volume     29983           0.296        4021         0.226                  0.499           0.951      1.000
striking / high volume   1449           0.014         440         0.209                  0.463           1.099      1.000
striking / mid volume   10387           0.102         619         0.259                  0.529           0.353      1.000
deep position           11207           0.110           0         0.311                  0.523             NaN      0.000
thin volume             39517           0.390           0         0.277      

### The decay / refresh insight — and what it does *not* say

The intuitive story is "old pages decay, refresh them". Two things in my own work say be careful with that.

1. **Age is a weak trigger in this portfolio.** ML-07 measured `content_age_days` as MIXED, and ML-09 §1
   showed why the paper's age curve cannot be read as a lifecycle: it is a snapshot of different pages, and
   the oldest bucket is survivors. So the playbook **does not use age to trigger a refresh.** Age appears as
   context in the export, never as a reason code.
2. **Momentum is the trigger that measured.** The Week-5 ablation dropped the ensemble from P@50 0.820 to
   0.516 without `log_momentum` — below the frozen rule. Whatever the model learned, it learned mostly from
   *whether the page was already fading inside March*.

The cell below turns that into the number the playbook is built on: among pages that were already fading, how
often did April actually fall below 80% of March, compared with pages that were steady? And it does the same
split inside the visible, ≥500-impression population the queue actually works on. That comparison is the
**observed** decay signal; it is one month pair, so it is a base rate to plan with, not a law.

In [3]:
vis = queue[queue.visible & (queue.impressions_30d >= VOLUME_FLOOR) & queue.observed_declined.notna()].copy()
vis["momentum_band"] = pd.cut(vis.log_momentum, [-100, -0.7, -0.35, 0.0, 0.35, 100],
                              labels=["falling hard (<-0.70)", "falling (-0.70..-0.35)",
                                      "soft (-0.35..0)", "steady (0..0.35)", "rising (>0.35)"])
band = vis.groupby("momentum_band", observed=True).agg(
    pages=("content_hash_id", "size"),
    observed_decline_rate=("observed_declined", "mean"),
    median_click_gap=("click_gap", "median"),
)
print("visible pages, >=500 impressions — observed April decline by March momentum:")
print(band.round(3).to_string())

fade = vis[vis.f_momentum]
steady = vis[~vis.f_momentum]
print(f"\nfading (log_momentum <= {MOMENTUM_CUT}): {len(fade):,} pages, "
      f"{fade.observed_declined.mean():.1%} declined in April")
print(f"steady (log_momentum >  {MOMENTUM_CUT}): {len(steady):,} pages, "
      f"{steady.observed_declined.mean():.1%} declined in April")
print(f"difference: {fade.observed_declined.mean() - steady.observed_declined.mean():+.1%} — "
      "observed association in one month pair, not a treatment effect.")

# the CTR gap is the other half of the story: does a gap predict decline on its own?
gap = vis[vis.f_ctr_gap]
nogap = vis[~vis.f_ctr_gap]
print(f"\nCTR below 70% of peer median: {len(gap):,} pages, {gap.observed_declined.mean():.1%} declined")
print(f"CTR at or above peer median : {len(nogap):,} pages, {nogap.observed_declined.mean():.1%} declined")
print(f"both signals (gap and fading): {len(vis[vis.f_ctr_gap & vis.f_momentum]):,} pages, "
      f"{vis[vis.f_ctr_gap & vis.f_momentum].observed_declined.mean():.1%} declined")
print("neither signal: ", end="")
neither = vis[~vis.f_ctr_gap & ~vis.f_momentum]
print(f"{len(neither):,} pages, {neither.observed_declined.mean():.1%} declined")

visible pages, >=500 impressions — observed April decline by March momentum:
                        pages  observed_decline_rate  median_click_gap
momentum_band                                                         
falling hard (<-0.70)    5446                  0.835             0.582
falling (-0.70..-0.35)   5629                  0.697             0.000
soft (-0.35..0)         10183                  0.577             0.000
steady (0..0.35)        11292                  0.452             0.000
rising (>0.35)          18103                  0.310             0.000

fading (log_momentum <= -0.35): 11,075 pages, 76.5% declined in April
steady (log_momentum >  -0.35): 39,578 pages, 41.9% declined in April
difference: +34.6% — observed association in one month pair, not a treatment effect.

CTR below 70% of peer median: 5,310 pages, 62.7% declined
CTR at or above peer median : 45,343 pages, 47.9% declined
both signals (gap and fading): 1,364 pages, 83.4% declined
neither signal: 35,632 

**Reading that honestly.** The two signals stack: pages with both a CTR gap and a fade declined most often,
pages with neither declined least, and each signal on its own sits in between. That is the whole basis for
the reason-code design, and it is a *base rate in one month pair*, not a mechanism. The playbook's promise is
"these pages look worth reviewing first", not "refreshing this page will recover its traffic".

It also explains why the playbook has **no age trigger**. If age were the driver, the refresh queue would be
sorted by `content_age_days`; the data says sort by momentum and CTR, and treat age as context.

## 2. Intended use and limits

**Intended use.** A weekly triage for an editor at a FlyRank client. The queue answers one question: *of the
pages we already know about, which fifty should a person look at this week, and why?* It is decision support
for a human workflow, and the output is a shortlist to review — not a list to execute.

**Who it is for.** Someone who can change a title tag and a page body, and who can say no. It assumes a
reviewer who knows the client's business, because the model does not.

**What it explicitly is not:**

| Not this | Why not |
|---|---|
| A traffic forecast | the label is a threshold on April vs March, and AUC is 0.64 — the score orders, it does not predict volume |
| A causal claim about refresh | nothing was randomised; refreshed pages are chosen pages |
| A portfolio-wide inventory | the frame is 57% of March pages, the ones with ≥100 impressions |
| A revenue or CPC model | neither exists in this release; `click_gap` is a click-sizing heuristic |
| A production service | a notebook, a CSV, and two figures. No scheduler, no API, no writes to any CMS |
| A per-page verdict | nine clients per held-out draw; the queue is a ranking, and rank 12 vs rank 19 is noise |
| A CTR verdict on a low-click page | below 5 expected clicks the ratio is a 0-vs-2 coin flip; those rows are watchlist, not work |

**The calibration sentence to keep next to the output.** In the grouped holdouts (ML-08/09), the model's
top 50 contained **82%** declining pages on average, ranging **62%–94%** across five draws. So a 50-page
batch is expected to contain roughly 41 real declines, and a bad draw could give 31. That is the number a
reviewer is buying when they work the head of the queue.

In [4]:
# ---- the limits, as numbers rather than adjectives
total_march = con.sql('''
SELECT COUNT(*) AS n FROM (SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp
FROM march WHERE gsc_data_available IS TRUE GROUP BY 1, 2)
''').df().n.iloc[0]

unlabelled = int(queue.observed_declined.isna().sum())
actionable = queue[queue.disposition == "actionable"]
top100 = queue.head(100)

print("population")
print(f"  March pages in the release          : {int(total_march):,}")
print(f"  described by the features (queue)   : {len(queue):,} ({len(queue) / total_march:.1%})")
print(f"  scored but never validated (no April): {unlabelled:,}")
print(f"  actionable                          : {len(actionable):,} ({len(actionable) / len(queue):.1%} of the queue)")

print("\nwhat a reviewer is buying at the head (from the grouped holdouts, not from this refit)")
print("  mean P@50 0.820 | range 0.62-0.94 across 5 draws -> expect ~41/50 declines, worst draw ~31/50")

print("\nconcentration")
share = top100.client_hash_id.value_counts()
b_share = batch.client_hash_id.value_counts()
print(f"  global top 100 : {len(share)} clients, largest {share.iloc[0]}% of the head - not client-fair")
print(f"  weekly batch   : {len(b_share)} clients, largest {b_share.iloc[0]}/{len(batch)} - the interleaved order")
print("  the batch fixes the allocation, not the model: a client with many zero-click pages still")
print("  dominates the global rank, which is why the trigger in section 4 watches this number.")

print("\nsizing coverage")
print(f"  click_gap defined for {queue.click_gap.notna().mean():.1%} of the queue")
print("  deep tiers have no peer median, so their gap is left empty rather than imputed;")
print("  every mean or sum of click_gap in this notebook is therefore a lower bound.")

print("\ntwo orders, one queue")
overlap = len(set(queue.head(50).content_hash_id) & set(
    queue.nsmallest(50, "rank_value").content_hash_id))
print(f"  risk-first top 50 vs clicks-per-hour top 50 overlap: {overlap}/50")
print("  where they disagree, the reviewer is choosing between 'most likely to fall' and 'most clicks at stake'.")

print("\nscore distribution of the actionable head")
print(actionable.head(200).model_score.describe().round(3).to_string())

population
  March pages in the release          : 176,738
  described by the features (queue)   : 101,441 (57.4%)
  scored but never validated (no April): 548
  actionable                          : 8,395 (8.3% of the queue)

what a reviewer is buying at the head (from the grouped holdouts, not from this refit)
  mean P@50 0.820 | range 0.62-0.94 across 5 draws -> expect ~41/50 declines, worst draw ~31/50

concentration
  global top 100 : 3 clients, largest 96% of the head - not client-fair
  weekly batch   : 25 clients, largest 3/50 - the interleaved order
  the batch fixes the allocation, not the model: a client with many zero-click pages still
  dominates the global rank, which is why the trigger in section 4 watches this number.

sizing coverage
  click_gap defined for 75.5% of the queue
  deep tiers have no peer median, so their gap is left empty rather than imputed;
  every mean or sum of click_gap in this notebook is therefore a lower bound.

two orders, one queue
  risk-first 

## 3. Human review + the no-go list

**Every row in the export has `review_required = True` and `auto_apply_allowed = False`.** That is not a
disclaimer at the bottom of a page; it is a column, and the notebook asserts it holds for all 101k rows.

**The review gate — seven checks before anyone edits a page:**

1. **Is the page live and indexable?** Nothing in this data proves it. A deindexed page is not a CTR problem.
2. **Is the tracking working?** If most of a client's pages are zero-click, the likelier explanation is a tag,
   property or consent problem than 200 bad titles. Every `ZERO_CLICKS` row starts with this check, not an edit.
3. **Is the peer the right peer?** `ctr_ratio` compares a page with its position tier across the whole
   portfolio. A branded page, a navigational page, or a page with an unusual SERP treatment may not belong
   to that peer group. If the reviewer cannot defend the comparison, the row is deferred.
4. **Does the intent explain the gap?** For `ZERO_CLICKS` and for `CTR_FAR_BELOW_PEERS`, read the query
   family first. A page ranking for a query it should not rank for has an intent problem, and no snippet
   rewrite fixes that.
5. **Does the page have business value?** The model has no revenue signal. A strategic page at rank 8 with a
   mediocre CTR may be worth more than a decaying page at 30,000 impressions — the reviewer decides.
6. **Is the batch capped?** 10 actions per client per week, 50 per week overall. A queue that demands 300
   edits a week is a queue nobody works.
7. **Is the decision logged?** Accept / defer / reject, with a one-line reason. That log is the only honest
   source of a future retraining label, and it is also how you find out the ranking is wrong.

**What should never be automated — the no-go list:**

| Never automate | Why |
|---|---|
| Publishing a content or title change | no reviewer, no accountability, and the model cannot read the page |
| Delete, redirect, canonical, noindex, or URL changes | irreversible, and outside what the features describe |
| Bulk rewrites from the queue | the head is 82% precise on average; the tail of a batch is where the misses are |
| Any action below the 500-impression floor | the label is noise there, and the effort is not worth it |
| Acting when the reason codes conflict | a deep page with a CTR gap is a relevance question, not a copy question |
| Client-facing claims of recovered traffic | nothing here was randomised; the queue is a shortlist |
| Re-fitting on the outcome month to sharpen *that* month's queue | that is the leakage the audit in ML-09 hunts |
| Ranking by revenue or CPC | neither column exists in this release |
| Refresh → rescore → refresh loops | a page can be edited into a worse CTR; a human has to see the result first |

In [5]:
def explain(r):
    '''The one sentence a reviewer reads before deciding.'''
    parts = []
    if r.f_ctr_far or r.f_ctr_gap:
        parts.append(f"CTR {r.ctr:.2f}% vs {r.peer_ctr:.2f}% peer median "
                     f"({r.ctr_ratio * 100:.0f}% of peer) on {r.impressions_30d:,.0f} impressions "
                     f"at position {r.avg_position_30d:.1f}")
    if r.f_momentum:
        parts.append(f"already fading in March (momentum {r.log_momentum:+.2f}, "
                     f"last ten days vs first ten)")
    if r.f_zero_click:
        parts.append(f"zero clicks where the peer median predicts {r.expected_clicks:.0f}")
    if r.f_low_expected:
        parts.append(f"only {r.expected_clicks:.1f} expected clicks — too few to read a CTR")
    if r.action_code == "WATCHLIST_EFFORT":
        parts.append(f"{r.impressions_30d:,.0f} impressions a month is below the effort floor "
                     f"for a two-hour edit")
    if r.f_deep:
        parts.append(f"deep at position {r.avg_position_30d:.1f} — relevance, not copy")
    if r.f_thin:
        parts.append(f"only {r.impressions_30d:,.0f} impressions — below the action floor")
    if not parts:
        parts.append("no flag fired")
    return "; ".join(parts) + "."


queue["why"] = [explain(r) for r in queue.itertuples()]
# rebuild the batch frame here so it carries the reviewer sentence
batch = queue[queue.in_weekly_batch].sort_values(["rank_in_client", "model_score"],
                                                 ascending=[True, False]).copy()
actionable = queue[queue.disposition == "actionable"].copy()

# ---- review preconditions, measured on the queue itself
actionable["needs_tracking_check"] = actionable.f_zero_click
actionable["signals_agree"] = actionable.f_ctr_gap & actionable.f_momentum
actionable["age_known"] = actionable.content_age_days > 0

top250 = actionable.head(250)
print("review preconditions across the top 250 actionable rows:")
print(f"  start with a tracking / intent check : {int(top250.needs_tracking_check.sum())}")
print(f"  both signals agree (higher confidence): {int(top250.signals_agree.sum())}")
print(f"  age known                           : {int(top250.age_known.sum())}")

print(f"\nthe weekly batch ({len(batch)} rows) after a cap of {CAP_PER_CLIENT} per client and client "
      f"interleaving: {batch.client_hash_id.nunique()} clients, {batch.effort_hours.sum():.1f} hours, "
      f"{batch.click_gap.sum():,.0f} clicks of measured gap")

print("\nfive rows as a reviewer would receive them:")
for r in batch.head(5).itertuples():
    gap = "not sized" if pd.isna(r.click_gap) else f"{r.click_gap:,.0f}"
    print(f"\n  #{r.rank} (client #{int(r.rank_in_client)})  {r.content_hash_id[:16]}..  |  "
          f"{r.action_code}  |  {r.effort_hours:.2f}h  |  click gap {gap}")
    print(f"      {r.why}")
    print(f"      action: {r.action}")

print("\nno-go enforcement, checked rather than promised:")
print(f"  rows with auto_apply_allowed = True: {int(queue.auto_apply_allowed.sum())}")
print(f"  rows with review_required = False : {int((~queue.review_required).sum())}")
print(f"  actionable rows below the volume floor: "
      f"{int((actionable.impressions_30d < VOLUME_FLOOR).sum())}")
assert queue.auto_apply_allowed.sum() == 0
assert (~queue.review_required).sum() == 0
assert (actionable.impressions_30d < VOLUME_FLOOR).sum() == 0
print("  -> all three checks pass; nothing in the export can act on its own.")

review preconditions across the top 250 actionable rows:
  start with a tracking / intent check : 3
  both signals agree (higher confidence): 129
  age known                           : 250

the weekly batch (50 rows) after a cap of 10 per client and client interleaving: 25 clients, 118.0 hours, 402 clicks of measured gap

five rows as a reviewer would receive them:

  #1 (client #1)  content_7a769ba6..  |  REFRESH_SNIPPET  |  3.00h  |  click gap 6
      CTR 0.00% vs 0.20% peer median (0% of peer) on 2,979 impressions at position 7.6; already fading in March (momentum -1.87, last ten days vs first ten).
      action: Refresh the body and rewrite the title/meta against the peer CTR gap.

  #179 (client #1)  content_08dca348..  |  REFRESH_BODY  |  2.00h  |  click gap 2
      already fading in March (momentum -1.64, last ten days vs first ten); only 4.1 expected clicks — too few to read a CTR.
      action: Refresh facts, examples and recrawl. CTR is not the measured problem.

  #207 (cli

## 4. Monitoring / retrain triggers

A queue built on one month pair goes stale in three ways: the **base rate moves** (the portfolio's decline
rate is not 51.5% forever), the **features move** (a tracking change flips `ctr` or `avg_position`), or the
**ranking stops matching the business** (reviewers keep rejecting the head).

The table below is the light version: every row is a number this notebook can compute today, a band to watch,
and what to do when the number leaves the band. Nothing here is a monitoring service — it is the checklist
someone runs when they regenerate the queue next month.

**Retrain when any of these is true for two consecutive months:**

- mean P@50 on the newest labelled month falls below **0.65** (measured 0.820 on grouped holdouts);
- the monthly decline base rate leaves **0.45–0.58** (today 0.515);
- the share of visible pages flagged as fading moves by more than **25% relative**;
- reviewers reject more than **50%** of the top 50 actions, or defer more than half;
- the decision moment or the feature windows change, or a new `content_type` appears;
- `gsc_data_available` coverage drops, or the April-style label is missing for a whole client.

**Do not retrain on the outcome month to improve that month's queue.** That is the leakage ML-09 tests for.
Retraining means: wait for a new labelled month, refit, and re-measure on grouped client holdouts.

In [6]:
visible_floor = queue[queue.visible & (queue.impressions_30d >= VOLUME_FLOOR)]
top100 = queue.head(100)
reason_mix = actionable.reason_primary.value_counts(normalize=True)

monitor = pd.DataFrame([
    {"metric": "monthly decline base rate", "current": round(float(labelled.declined.mean()), 3),
     "watch band": "0.45 - 0.58", "what it means": "the label's prevalence; a shift changes every precision number"},
    {"metric": "share of visible pages fading (momentum <= -0.35)",
     "current": round(float(visible_floor.f_momentum.mean()), 3), "watch band": "+/- 25% relative",
     "what it means": "the feature that carries the model; a tracking change would move it"},
    {"metric": "median CTR ratio, visible pages",
     "current": round(float(visible_floor.ctr_ratio.median()), 3), "watch band": "+/- 20% relative",
     "what it means": "peer medians are recomputed from this population every month"},
    {"metric": "top-100 client concentration",
     "current": round(float(top100.client_hash_id.value_counts().iloc[0] / 100), 3),
     "watch band": "< 0.40", "what it means": "one client dominating the head means the ranking is not transferable"},
    {"metric": "largest single reason share",
     "current": round(float(reason_mix.iloc[0]), 3), "watch band": "< 0.85",
     "what it means": "momentum is expected to dominate after the CTR gate; a CTR code taking over means the peer medians moved"},
    {"metric": "queue rows with no April label",
     "current": round(float(queue.observed_declined.isna().mean()), 4), "watch band": "< 0.02",
     "what it means": "scored but never validated; a rising share means the label join is breaking"},
    {"metric": "actionable share of the queue",
     "current": round(float((queue.disposition == "actionable").mean()), 3), "watch band": "0.05 - 0.25",
     "what it means": "too high and the queue is not prioritising; too low and the thresholds are too strict"},
    {"metric": "share of visible pages below 5 expected clicks",
     "current": round(float((queue.visible & queue.f_low_expected).sum() / max(int(queue.visible.sum()), 1)), 3),
     "watch band": "informational",
     "what it means": "a rising share means the queue is drifting toward CTRs too small to read"},
    {"metric": "mean P@50 on grouped holdouts", "current": 0.820, "watch band": ">= 0.65",
     "what it means": "the only out-of-sample quality number this playbook has"},
])
print("monitoring table — current value, band, meaning:")
print(monitor.to_string(index=False))

top_client = top100.client_hash_id.value_counts().index[0]
tc_vis = queue[(queue.client_hash_id == top_client) & queue.ctr_ratio.notna()]
print(f"\none guard is already outside its band on day one: top-100 concentration "
      f"{float(monitor.loc[monitor.metric == 'top-100 client concentration', 'current'].iloc[0]):.2f} "
      f"vs a band of < 0.40.")
print(f"  the dominant client supplies {int(top100.client_hash_id.value_counts().iloc[0])}/100 of the head, "
      f"{int((tc_vis.clicks_30d == 0).sum())}/{len(tc_vis)} of its visible pages have zero clicks, "
      f"and it gets {int(batch.client_hash_id.eq(top_client).sum())}/{len(batch)} rows in the batch.")
print("  that is a finding, not a bug: zero-click pages dominate the model's head because zero clicks is")
print("  the strongest feature the model has, and a portfolio-wide rank is not client-fair. The batch")
print("  interleaves clients and the cap does the rest; the threshold itself is the thing to watch.")

print("\nretrain triggers (any two months in a row):")
for line in [
    "P@50 below 0.65 on the newest labelled month",
    "base rate outside 0.45-0.58",
    "fading share moves more than 25% relative",
    "reviewers reject >50% or defer >50% of the top 50",
    "decision moment or feature windows change; new content_type appears",
    "label coverage drops for a whole client or a whole month",
]:
    print(f"  - {line}")

print("\nwhat this notebook does NOT do: no scheduler, no API, no writes outside work/, no retraining loop.")

monitoring table — current value, band, meaning:
                                           metric  current       watch band                                                                                            what it means
                        monthly decline base rate   0.5150      0.45 - 0.58                                           the label's prevalence; a shift changes every precision number
share of visible pages fading (momentum <= -0.35)   0.2190 +/- 25% relative                                      the feature that carries the model; a tracking change would move it
                  median CTR ratio, visible pages   1.1990 +/- 20% relative                                             peer medians are recomputed from this population every month
                     top-100 client concentration   0.9600           < 0.40                                     one client dominating the head means the ranking is not transferable
                      largest single reason sh

## 5. Exports for the paper

The notebook regenerates four things, and the paper will build on exactly these:

| File | What it is | In git? |
|---|---|---|
| `work/outputs/action_queue.csv` | the full ranked queue, every page the features can describe, with reason, action, effort, disposition and review flags | **no** — `work/**/*.csv` is gitignored by design, and the notebook regenerates it |
| `work/outputs/w07_playbook_summary.json` | the receipt: population, thresholds, dispositions, reason mix, archetype base rates, decay comparison, holdout precision, monitoring table | yes |
| `work/figures/playbook_value_and_cost.png` | the same queue in two orders — model risk vs clicks-per-hour: where the measured gap is, and what each order costs in hours | yes |
| `work/figures/playbook_reason_mix.png`, `playbook_archetype_outcome.png`, `playbook_momentum_decay.png` | what the queue is made of, and the decay evidence behind the reason codes | yes |

**The weekly batch.** The full queue is for analysis; a person works a batch. Section 1 already built it —
interleaved across clients, capped at 10 per client and 50 per week — and the export marks it with
`in_weekly_batch`, so the practical plan and the research artifact come from the same file and cannot drift
apart.

**One rule for anything downstream.** Every number in the summary JSON is either (a) computed from this
release and labelled as observed, or (b) copied from the ML-08/09 grouped holdouts and labelled as
out-of-sample. The in-sample refit's own precision is never written into the JSON, because a model scored on
the rows it was fitted on cannot produce one.

In [7]:
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def repo_root():
    '''Walk up until work/notebooks exists, so the notebook works from anywhere.'''
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "work" / "notebooks").is_dir():
            return p
    return Path.cwd()


ROOT = repo_root()
OUT = ROOT / "work" / "outputs"
FIG = ROOT / "work" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

# ---- the weekly batch was built in section 1 (interleaved, capped); export it as a flag
print(f"weekly batch: {len(batch)} actions | {batch.client_hash_id.nunique()} clients | "
      f"{batch.effort_hours.sum():.1f} hours | {batch.click_gap.sum():,.0f} clicks of measured gap")
print(batch.action_code.value_counts().to_string())

# ---- export the queue
EXPORT_COLS = [
    "rank", "in_weekly_batch", "disposition", "action_code", "action",
    "reason_primary", "reason_secondary", "why", "archetype",
    "client_hash_id", "content_hash_id", "content_type",
    "model_score", "impressions_30d", "clicks_30d", "ctr", "peer_ctr", "ctr_ratio",
    "expected_clicks", "avg_position_30d", "position_tier", "days_with_impressions_30d",
    "content_age_days", "click_gap", "effort_hours", "hours_if_worked", "value_per_hour",
    "review_required", "auto_apply_allowed", "observed_declined",
]
export = queue[EXPORT_COLS].copy()
export["position_tier"] = export.position_tier.astype(str)
export["observed_declined"] = export.observed_declined.astype("Int64")
for col, nd in [("model_score", 4), ("ctr", 3), ("peer_ctr", 3), ("ctr_ratio", 3),
                ("expected_clicks", 2), ("avg_position_30d", 2), ("click_gap", 1),
                ("value_per_hour", 2)]:
    export[col] = export[col].round(nd)

QUEUE_CSV = OUT / "action_queue.csv"
export.to_csv(QUEUE_CSV, index=False)

# ---- the receipt
summary = {
    "artifact": "ML-10 content action playbook",
    "lane": "Lane 4 - CTR / engagement opportunity scoring",
    "decision_moment": "2026-03-31",
    "label_window": "2026-04 (April impressions < 80% of March)",
    "frame": {
        "march_pages_in_release": int(total_march),
        "pages_described_by_features": int(len(queue)),
        "labelled_rows_used_for_refit": int(len(labelled)),
        "scored_but_never_validated": int(unlabelled),
        "clients_in_queue": int(queue.client_hash_id.nunique()),
        "clients_in_label": int(labelled.client_hash_id.nunique()),
        "share_of_march_pages_covered": round(len(queue) / int(total_march), 4),
        "observed_decline_base_rate": round(float(labelled.declined.mean()), 4),
    },
    "thresholds": {
        "volume_floor_impressions": VOLUME_FLOOR,
        "momentum_cut_log_ratio": MOMENTUM_CUT,
        "ctr_gap_cut_ratio": CTR_GAP_CUT,
        "ctr_severe_cut_ratio": CTR_SEVERE_CUT,
        "min_expected_clicks": MIN_EXPECTED_CLICKS,
        "effort_floor_impressions": EFFORT_FLOOR,
        "cap_per_client_per_week": CAP_PER_CLIENT,
        "weekly_batch_size": BATCH_SIZE,
        "no_age_trigger": True,
    },
    "dispositions": {k: int(v) for k, v in queue.disposition.value_counts().items()},
    "actionable_share_of_queue": round(float((queue.disposition == "actionable").mean()), 4),
    "reason_mix_actionable": {k: int(v) for k, v in actionable.reason_primary.value_counts().items()},
    "archetypes": {
        name: {
            "pages": int(row.pages),
            "share_of_queue": float(row.share_of_queue),
            "fading_share": round(float(row.fading_share), 4),
            "observed_decline_rate": round(float(row.observed_decline_rate), 4),
            "mean_click_gap": None if pd.isna(row.mean_click_gap) else round(float(row.mean_click_gap), 2),
            "share_sized": round(float(row.pct_sized), 4),
            "mapped_action": row.mapped_action,
        }
        for name, row in arch.iterrows()
    },
    "decay_evidence": {
        "fading_pages": int(len(fade)),
        "fading_decline_rate": round(float(fade.observed_declined.mean()), 4),
        "steady_pages": int(len(steady)),
        "steady_decline_rate": round(float(steady.observed_declined.mean()), 4),
        "difference": round(float(fade.observed_declined.mean() - steady.observed_declined.mean()), 4),
        "both_signals_pages": int(len(vis[vis.f_ctr_gap & vis.f_momentum])),
        "both_signals_decline_rate": round(float(vis[vis.f_ctr_gap & vis.f_momentum].observed_declined.mean()), 4),
        "neither_signal_pages": int(len(neither)),
        "neither_signal_decline_rate": round(float(neither.observed_declined.mean()), 4),
        "by_momentum_band": {
            str(k): {"pages": int(v.pages), "observed_decline_rate": round(float(v.observed_decline_rate), 4)}
            for k, v in band.iterrows()
        },
        "note": "one month pair; observed association, not a treatment effect; age is not used as a trigger",
    },
    "holdout_quality": {
        "source": ["work/outputs/w05_model_metrics.json", "work/outputs/w06_validation_audit.json"],
        "design": "GroupShuffleSplit by client_hash_id, 20% of clients held out, seeds 42/7/2024/101/555",
        "hgb_precision_at_50_mean": 0.820,
        "hgb_precision_at_50_min": 0.62,
        "hgb_precision_at_50_max": 0.94,
        "hgb_roc_auc_grouped": 0.643,
        "frozen_rule_precision_at_50": 0.540,
        "hgb_without_log_momentum_precision_at_50": 0.516,
        "note": "out-of-sample only; the in-sample refit in this notebook produces no precision number",
    },
    "weekly_batch": {
        "actions": int(len(batch)),
        "clients": int(batch.client_hash_id.nunique()),
        "hours": round(float(batch.effort_hours.sum()), 1),
        "measured_click_gap": round(float(batch.click_gap.sum()), 1),
        "action_mix": {k: int(v) for k, v in batch.action_code.value_counts().items()},
    },
    "monitoring": monitor.to_dict(orient="records"),
    "no_go": [
        "publishing a content or title change",
        "delete, redirect, canonical, noindex or URL changes",
        "bulk rewrites from the queue",
        "any action below the 500-impression floor",
        "acting when the reason codes conflict",
        "client-facing claims of recovered traffic",
        "refitting on the outcome month to sharpen that month's queue",
        "ranking by revenue or CPC (neither column exists)",
        "refresh -> rescore -> refresh loops without a human in between",
    ],
    "exports": {
        "queue_csv": "work/outputs/action_queue.csv",
        "summary_json": "work/outputs/w07_playbook_summary.json",
        "figures": ["work/figures/playbook_value_and_cost.png",
                    "work/figures/playbook_reason_mix.png",
                    "work/figures/playbook_archetype_outcome.png",
                    "work/figures/playbook_momentum_decay.png"],
        "queue_csv_in_git": False,
    },
}

SUMMARY_JSON = OUT / "w07_playbook_summary.json"
SUMMARY_JSON.write_text(json.dumps(summary, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                        encoding="utf-8")

print(f"\nwrote {QUEUE_CSV.relative_to(ROOT)}  ({QUEUE_CSV.stat().st_size / 1e6:.1f} MB, {len(export):,} rows)")
print(f"wrote {SUMMARY_JSON.relative_to(ROOT)}  ({SUMMARY_JSON.stat().st_size / 1e3:.1f} kB)")

weekly batch: 50 actions | 25 clients | 118.0 hours | 402 clicks of measured gap
action_code
REFRESH_BODY       28
REFRESH_SNIPPET    20
SNIPPET_REWRITE     2



wrote work\outputs\action_queue.csv  (39.6 MB, 101,441 rows)
wrote work\outputs\w07_playbook_summary.json  (7.9 kB)


### The four figures

Each one exists to be reused in the paper, so each one carries its own caption in the code below: what is
observed, and what the reader must not read into it.

In [8]:
BLUE, AMBER, GREY, RED = "#1f6feb", "#c69026", "#8b949e", "#c2413a"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.labelsize": 9, "font.size": 9})

act_risk = actionable.sort_values(["model_score", "click_gap"], ascending=False).reset_index(drop=True)
act_value = actionable.sort_values("value_per_hour", ascending=False).reset_index(drop=True)
N_PLOT = 400


def curves(df):
    gap = df.click_gap.fillna(0).cumsum()
    return gap / gap.iloc[-1] * 100, df.effort_hours.cumsum()


gap_risk, hours_risk = curves(act_risk)
gap_value, hours_value = curves(act_value)
ranks = np.arange(1, N_PLOT + 1)

# ---- figure 1: the same queue in two orders - risk first vs clicks-per-hour first
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.1))
ax = axes[0]
ax.plot(ranks, gap_risk[:N_PLOT], color=BLUE, lw=2, label="model risk order")
ax.plot(ranks, gap_value[:N_PLOT], color=AMBER, lw=2, label="clicks-per-hour order")
ax.axvline(50, color=GREY, ls="--", lw=1)
ax.plot([50, 50], [gap_risk.iloc[49], gap_value.iloc[49]], "o", color=GREY, ms=4)
ax.annotate(f"risk order, top 50: {gap_risk.iloc[49]:.0f}%", xy=(50, gap_risk.iloc[49]),
            xytext=(105, gap_risk.iloc[49] + 2), fontsize=8.5,
            arrowprops={"arrowstyle": "->", "color": GREY, "lw": 0.8})
ax.annotate(f"value order, top 50: {gap_value.iloc[49]:.0f}%", xy=(50, gap_value.iloc[49]),
            xytext=(105, gap_value.iloc[49] + 8), fontsize=8.5,
            arrowprops={"arrowstyle": "->", "color": GREY, "lw": 0.8})
ax.set_xlabel("rows worked, in each order")
ax.set_ylabel("cumulative share of measured click gap (%)")
ax.set_title("Where the measured clicks are")
ax.set_ylim(0, 102)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")

ax = axes[1]
ax.plot(ranks, hours_risk[:N_PLOT], color=BLUE, lw=2)
ax.plot(ranks, hours_value[:N_PLOT], color=AMBER, lw=2)
ax.axvline(50, color=GREY, ls="--", lw=1)
ax.annotate(f"risk order: {hours_risk.iloc[49]:.0f}h", xy=(50, hours_risk.iloc[49]),
            xytext=(130, hours_risk.iloc[49] + 60), fontsize=8.5,
            arrowprops={"arrowstyle": "->", "color": GREY, "lw": 0.8})
ax.annotate(f"value order: {hours_value.iloc[49]:.0f}h", xy=(50, hours_value.iloc[49]),
            xytext=(130, hours_value.iloc[49] + 10), fontsize=8.5,
            arrowprops={"arrowstyle": "->", "color": GREY, "lw": 0.8})
ax.set_xlabel("rows worked, in each order")
ax.set_ylabel("cumulative editor hours")
ax.set_title("What each order costs")
fig.suptitle("Figure 1 - One queue, two orders: the model's head is not where the clicks are "
             f"({gap_risk.iloc[49]:.0f}% vs {gap_value.iloc[49]:.0f}% of the gap in the first 50)",
             fontsize=10.5, y=1.02)
fig.tight_layout()
fig.savefig(FIG / "playbook_value_and_cost.png", dpi=140, bbox_inches="tight")
plt.close(fig)

# ---- figure 2: what the queue is made of, and which action each reason prescribes
mix = actionable.reason_primary.value_counts()
pivot = (actionable.pivot_table(index="reason_primary", columns="action_code",
                                values="content_hash_id", aggfunc="count", fill_value=0)
         .reindex(mix.index[::-1]))
ACTION_COLORS = {"REFRESH_SNIPPET": BLUE, "SNIPPET_REWRITE": "#7aa7f0", "REFRESH_BODY": AMBER,
                 "SNIPPET_TEST": "#e3bd72", "INTENT_CHECK": RED}
fig, ax = plt.subplots(figsize=(9.6, 4.2))
ys = np.arange(len(pivot))
left = np.zeros(len(pivot))
for action in pivot.sum().sort_values(ascending=False).index:
    vals = pivot[action].values
    ax.barh(ys, vals, left=left, label=action, color=ACTION_COLORS.get(action, GREY), alpha=0.9)
    left += vals
for y, total in zip(ys, pivot.sum(axis=1).values):
    ax.text(total * 1.01, y, f"{total:,}", va="center", fontsize=8.5)
ax.set_yticks(ys)
ax.set_yticklabels(pivot.index)
ax.set_xlabel("actionable pages")
ax.set_title("Figure 2 - Reason code drives the action: what the actionable queue is made of")
ax.set_xlim(0, pivot.sum(axis=1).max() * 1.12)
ax.legend(frameon=False, fontsize=8.5, ncol=3, loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "playbook_reason_mix.png", dpi=140, bbox_inches="tight")
plt.close(fig)

# ---- figure 3: archetype size vs observed outcome
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
ys = np.arange(len(arch))[::-1]
ax = axes[0]
ax.barh(ys, arch.pages.values, color=BLUE, alpha=0.85)
for y, v in zip(ys, arch.pages.values):
    ax.text(v * 1.01, y, f"{v:,}", va="center", fontsize=8.5)
ax.set_yticks(ys)
ax.set_yticklabels(arch.index)
ax.set_xlabel("pages in the queue")
ax.set_title("Queue composition")
ax.set_xlim(0, arch.pages.max() * 1.16)

ax = axes[1]
ax.barh(ys, arch.observed_decline_rate.values * 100, color=AMBER, alpha=0.9)
base = float(labelled.declined.mean()) * 100
ax.axvline(base, color=GREY, ls="--", lw=1)
ax.set_ylim(-0.6, len(arch) - 0.2)
ax.text(base, len(arch) - 0.45, f"base rate {base:.1f}%", fontsize=8, color=GREY,
        va="bottom", ha="center")
for y, v in zip(ys, arch.observed_decline_rate.values * 100):
    ax.text(v + 1.0, y, f"{v:.1f}%", va="center", fontsize=8.5)
ax.set_yticks(ys)
ax.set_yticklabels([])
ax.set_xlabel("observed April decline rate (%)")
ax.set_title("Observed outcome per archetype")
ax.set_xlim(0, max(arch.observed_decline_rate.max() * 100, base) * 1.25)
fig.suptitle("Figure 3 - Archetype -> action map, with the observed April decline rate as a base rate, not a promise",
             fontsize=10.5, y=1.02)
fig.tight_layout()
fig.savefig(FIG / "playbook_archetype_outcome.png", dpi=140, bbox_inches="tight")
plt.close(fig)

# ---- figure 4: the decay evidence behind MOMENTUM_FADE
fig, ax = plt.subplots(figsize=(9.0, 4.0))
b_vals = band.observed_decline_rate.values * 100
b_n = band.pages.values
xs = np.arange(len(band))
bars = ax.bar(xs, b_vals, color=[RED if i < 2 else (AMBER if i == 2 else BLUE) for i in range(len(band))],
              alpha=0.9)
ax.axhline(base, color=GREY, ls="--", lw=1)
ax.text(len(band) - 0.55, base + 1.2, f"portfolio base rate {base:.1f}%", fontsize=8, color=GREY,
        ha="right", va="bottom")
for x, v, n in zip(xs, b_vals, b_n):
    ax.text(x, v + 0.6, f"{v:.1f}%", ha="center", fontsize=9)
    ax.text(x, 1.0, f"n={n:,}", ha="center", fontsize=7.5, color="white")
ax.set_xticks(xs)
ax.set_xticklabels([str(c).replace(" (", "\n(") for c in band.index], fontsize=8)
ax.set_ylabel("observed April decline rate (%)")
ax.set_ylim(0, max(b_vals.max(), base) * 1.18)
ax.set_title("Figure 4 - Fading pages declined more often: the signal the playbook is built on (one month pair)")
fig.tight_layout()
fig.savefig(FIG / "playbook_momentum_decay.png", dpi=140, bbox_inches="tight")
plt.close(fig)

for name in ["playbook_value_and_cost", "playbook_reason_mix",
             "playbook_archetype_outcome", "playbook_momentum_decay"]:
    p = FIG / f"{name}.png"
    print(f"wrote {p.relative_to(ROOT)}  ({p.stat().st_size / 1e3:.0f} kB)")

wrote work\figures\playbook_value_and_cost.png  (101 kB)
wrote work\figures\playbook_reason_mix.png  (38 kB)
wrote work\figures\playbook_archetype_outcome.png  (62 kB)
wrote work\figures\playbook_momentum_decay.png  (50 kB)


## 6. Self-check

The playbook is only trustworthy if the claims in it can be traced. This section re-reads the exported files
and checks them against the numbers the notebook just printed, so a reader does not have to take any of the
prose on faith.

| Claim in this notebook | How it is checked | Result |
|---|---|---|
| every row needs a human | `review_required` is `True` and `auto_apply_allowed` is `False` on all rows of the exported CSV | see cell below |
| nothing below the volume floor is actionable | `actionable.impressions_30d >= 500`, asserted | see cell below |
| CTR codes only fire where the CTR is readable | every `CTR_*` row has at least 5 expected clicks | see cell below |
| no two-hour edit lands on a low-volume page | every action of 2h or more has at least 2,000 impressions | see cell below |
| the queue is reproducible from the notebook | the CSV is regenerated here and its row count matches the in-memory frame | see cell below |
| precision claims are out-of-sample | the JSON's `holdout_quality` cites the ML-08/09 files and the design; no in-sample precision is written | see cell below |
| age is not a trigger | no reason code reads `content_age_days`; it appears only as context | see cell below |
| no client identity leaks | only `client_hash_id` / `content_hash_id`, no names, URLs or queries | see cell below |
| the batch is client-fair while the rank is not | batch client count vs top-100 concentration, both measured | see cell below |
| zero clicks lead to a check, not a rewrite | every `ZERO_CLICKS` row carries `INTENT_CHECK` | see cell below |
| the plan is non-production | the batch is 50 rows, capped at 10 per client, and nothing writes outside `work/` | see cell below |

**What I would do differently with another month.** Three things are worth naming rather than hiding. First,
the global head is **96 of the top 100 from one client**, all of it zero-click: a portfolio-wide ranker is not
client-fair, and the fix belongs in the batch construction (done here) and in a tracking check at onboarding
(not done here). Second, 548 pages in the queue have no April row — scored but never validated. Third,
`ctr_ratio` is missing for about a quarter of pages (the deep tiers), so their click gap is left empty rather
than imputed, and every mean or sum of `click_gap` in this notebook is a lower bound. All three are visible in
the export rather than smoothed away, which is the point of exporting the queue instead of a top-50 list.

In [9]:
# ---- read the exports back and check them against the claims
back = pd.read_csv(QUEUE_CSV, dtype={"client_hash_id": str, "content_hash_id": str})

checks = []


def check(label, passed, detail):
    checks.append({"check": label, "result": "PASS" if passed else "FAIL", "detail": detail})


check("queue row count matches the in-memory frame", len(back) == len(queue),
      f"{len(back):,} rows written vs {len(queue):,} in memory")
check("every exported row requires review", bool(back.review_required.all()),
      f"{int(back.review_required.sum()):,}/{len(back):,} review_required = True")
check("no exported row may act on its own", not bool(back.auto_apply_allowed.any()),
      f"{int(back.auto_apply_allowed.sum())} rows with auto_apply_allowed = True")
check("no actionable row below the volume floor",
      int(back[(back.disposition == "actionable") & (back.impressions_30d < VOLUME_FLOOR)].shape[0]) == 0,
      f"floor = {VOLUME_FLOOR} impressions")
ctr_prim = back[back.reason_primary.isin(["CTR_FAR_BELOW_PEERS", "CTR_BELOW_PEERS"])]
check("CTR codes only where the CTR is readable",
      bool((ctr_prim.expected_clicks >= MIN_EXPECTED_CLICKS).all()),
      f"{len(ctr_prim):,} CTR rows, minimum {float(ctr_prim.expected_clicks.min()):.1f} expected clicks")
check("no two-hour edit below the effort floor",
      int(back[(back.effort_hours >= 2) & (back.impressions_30d < EFFORT_FLOOR)].shape[0]) == 0,
      f"floor = {EFFORT_FLOOR:,} impressions for actions of 2h or more")
check("dispositions are only the three agreed values",
      set(back.disposition.unique()) <= {"actionable", "watchlist", "no_action"},
      ", ".join(sorted(back.disposition.unique())))
check("every row carries a reason and an action",
      bool(back.reason_primary.notna().all() and back.action_code.notna().all() and back.why.notna().all()),
      f"{back.reason_primary.nunique()} distinct primary reasons, {back.action_code.nunique()} actions")
check("weekly batch is 50 rows, at most 10 per client",
      int(back.in_weekly_batch.sum()) == BATCH_SIZE
      and int(back[back.in_weekly_batch].client_hash_id.value_counts().max()) <= CAP_PER_CLIENT,
      f"{int(back.in_weekly_batch.sum())} rows, largest client "
      f"{int(back[back.in_weekly_batch].client_hash_id.value_counts().max())}")
check("the batch is client-fair, the global rank is not",
      back[back.in_weekly_batch].client_hash_id.nunique() >= 20
      and int(back.head(100).client_hash_id.value_counts().iloc[0]) > 40,
      f"batch covers {back[back.in_weekly_batch].client_hash_id.nunique()} clients; "
      f"global top 100 has {int(back.head(100).client_hash_id.value_counts().iloc[0])}/100 from one client")
check("zero-click rows prescribe a check, not a rewrite",
      set(back.loc[back.reason_primary == "ZERO_CLICKS", "action_code"]) == {"INTENT_CHECK"},
      f"{int((back.reason_primary == 'ZERO_CLICKS').sum()):,} ZERO_CLICKS rows, "
      f"all -> {', '.join(sorted(set(back.loc[back.reason_primary == 'ZERO_CLICKS', 'action_code'])))}")
check("only hashed identifiers are exported",
      all(c in {"client_hash_id", "content_hash_id"} for c in back.columns
          if c.endswith("_id")) and back.client_hash_id.str.startswith("client_").all(),
      "client_hash_id / content_hash_id only")
check("no URL, query or client name column",
      not any(t in c.lower() for c in back.columns for t in ("url", "query", "domain", "client_name")),
      f"{len(back.columns)} columns, none free-text")
check("age is context, not a trigger",
      "content_age_days" not in (" ".join(back.reason_primary.dropna().unique())
                                 + " ".join(back.reason_secondary.dropna().unique())),
      "no reason code references content_age_days")
check("summary JSON parses and cites the holdouts",
      json.loads(SUMMARY_JSON.read_text(encoding="utf-8"))["holdout_quality"]["hgb_precision_at_50_mean"] == 0.820,
      "P@50 0.820 copied from the grouped holdouts, not from this refit")
check("figures exist and are non-trivial", all((FIG / f).stat().st_size > 20_000
      for f in ["playbook_value_and_cost.png", "playbook_reason_mix.png",
                "playbook_archetype_outcome.png", "playbook_momentum_decay.png"]),
      "4 PNGs in work/figures/")

report = pd.DataFrame(checks)
print(report.to_string(index=False))
print(f"\n{int((report.result == 'PASS').sum())}/{len(report)} checks passed")

assert (report.result == "PASS").all(), "a self-check failed - fix before exporting"

print("\nexports (regenerate by re-running this notebook):")
for name, path in [("queue", QUEUE_CSV), ("summary", SUMMARY_JSON)]:
    print(f"  {name:<8} {path.relative_to(ROOT)}  {path.stat().st_size / 1e3:>8.0f} kB")
for f in sorted(FIG.glob("playbook_*.png")):
    print(f"  figure   {f.relative_to(ROOT)}  {f.stat().st_size / 1e3:>8.0f} kB")
print("\nnon-production by construction: no scheduler, no API, no CMS writes, no retraining loop.")

                                           check result                                                             detail
     queue row count matches the in-memory frame   PASS                          101,441 rows written vs 101,441 in memory
              every exported row requires review   PASS                             101,441/101,441 review_required = True
              no exported row may act on its own   PASS                              0 rows with auto_apply_allowed = True
        no actionable row below the volume floor   PASS                                            floor = 500 impressions
        CTR codes only where the CTR is readable   PASS                        5,249 CTR rows, minimum 5.0 expected clicks
         no two-hour edit below the effort floor   PASS                floor = 2,000 impressions for actions of 2h or more
   dispositions are only the three agreed values   PASS                                   actionable, no_action, watchlist
        every ro